In [ ]:
import os, gc, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import torch.backends.cudnn as cudnn

from torchvision import datasets, transforms
from torchvision.datasets.folder import default_loader
import timm

from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# -----------------------------
# Config
# -----------------------------
class CFG:
    base_dir  = "/kaggle/input/srifoton-25-machine-learning-competition"
    train_dir = os.path.join(base_dir, "train")
    val_dir   = os.path.join(base_dir, "val")
    test_dir  = os.path.join(base_dir, "test")

    model_name = "tf_efficientnet_b4_ns"   
    img_size   = None                      
    batch_size = 16
    num_workers= 2
    epochs     = 10
    lr         = 2e-4
    weight_decay = 1e-4
    seed       = 42
    use_weighted_loss = True
    tta = 0

    label_map = {
        "Bacterial Pneumonia": 0,
        "Corona Virus Disease": 1,
        "Normal": 2,
        "Tuberculosis": 3,
        "Viral Pneumonia": 4
    }

# -----------------------------
# Fix path if nested
# -----------------------------
def resolve_path(path):
    # kalau di dalamnya hanya ada 1 folder bernama sama, masuk ke dalam
    items = os.listdir(path)
    if len(items) == 1 and os.path.isdir(os.path.join(path, items[0])):
        return os.path.join(path, items[0])
    return path

CFG.train_dir = resolve_path(CFG.train_dir)
CFG.val_dir   = resolve_path(CFG.val_dir)
CFG.test_dir  = resolve_path(CFG.test_dir)

print("Train dir:", CFG.train_dir)
print("Val dir:", CFG.val_dir)
print("Test dir:", CFG.test_dir)


In [ ]:
# -----------------------------
# Seed & Device
# -----------------------------
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = False
    cudnn.benchmark = True

seed_everything(CFG.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# -----------------------------
# Input size dari timm
# -----------------------------
def resolve_size_from_timm(model_name: str):
    tmp = timm.create_model(model_name, pretrained=True)
    sz = tmp.default_cfg.get("input_size", (3,224,224))[1]
    del tmp; gc.collect()
    return sz

if CFG.img_size is None:
    CFG.img_size = resolve_size_from_timm(CFG.model_name)
print("Input size:", CFG.img_size)

In [ ]:
# -----------------------------
# Transforms
# -----------------------------
train_tfms = transforms.Compose([
    transforms.Resize((CFG.img_size, CFG.img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_tfms = transforms.Compose([
    transforms.Resize((CFG.img_size, CFG.img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [ ]:
# -----------------------------
# Datasets & Loaders
# -----------------------------
train_ds = datasets.ImageFolder(CFG.train_dir, transform=train_tfms)
val_ds   = datasets.ImageFolder(CFG.val_dir,   transform=val_tfms)

class_names = sorted([c for c in train_ds.classes if c in CFG.label_map])
print("Detected classes:", class_names)

# Weighted sampler (optional)
if CFG.use_weighted_loss:
    counts = np.zeros(len(train_ds.classes), dtype=np.int64)
    for _, t in train_ds.samples:
        counts[t] += 1
    class_weights = 1.0 / np.clip(counts, 1, None)
    sample_weights = [class_weights[label] for _, label in train_ds.samples]
    sampler = WeightedRandomSampler(weights=sample_weights,
                                    num_samples=len(sample_weights),
                                    replacement=True)
    shuffle_train = False
else:
    sampler = None
    shuffle_train = True

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size,
                          shuffle=shuffle_train, sampler=sampler,
                          num_workers=CFG.num_workers, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=CFG.batch_size,
                          shuffle=False, num_workers=CFG.num_workers, pin_memory=True)

In [ ]:
# -----------------------------
# Model
# -----------------------------
def build_model(model_name: str, num_classes: int):
    model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
    return model

model = build_model(CFG.model_name, num_classes=len(class_names)).to(device)

# Weighted CE Loss
if CFG.use_weighted_loss:
    counts = np.zeros(len(train_ds.classes), dtype=np.int64)
    for _, t in train_ds.samples:
        counts[t] += 1
    cw = (1.0 / np.clip(counts, 1, None))
    cw = cw / cw.sum() * len(train_ds.classes)
    cw = torch.tensor(cw, dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw)
else:
    criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs)

In [ ]:
# -----------------------------
# Train & Eval
# -----------------------------
def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            preds = out.argmax(1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
    f1 = f1_score(y_true, y_pred, average="macro")
    return f1, classification_report(y_true, y_pred, target_names=class_names), confusion_matrix(y_true, y_pred)

def train_one_epoch(model, loader):
    model.train()
    running = 0.0
    for x, y in tqdm(loader, leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running += loss.item() * x.size(0)
    return running / len(loader.dataset)

best_f1 = -1
for epoch in range(1, CFG.epochs+1):
    tr_loss = train_one_epoch(model, train_loader)
    val_f1, val_report, cm = evaluate(model, val_loader)
    scheduler.step()

    print(f"\nEpoch {epoch}/{CFG.epochs} | TrainLoss: {tr_loss:.4f} | Val F1: {val_f1:.4f}")
    print(val_report)

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "best_model.pth")
        print("✅ Best model saved!")

print("Best Val Macro F1:", best_f1)

In [ ]:
# -----------------------------
# Inference Test
# -----------------------------
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

test_images = sorted([f for f in os.listdir(CFG.test_dir) if f.lower().endswith(".jpeg")])
results = []
with torch.no_grad():
    for img_name in tqdm(test_images):
        path = os.path.join(CFG.test_dir, img_name)
        img = default_loader(path)
        x = val_tfms(img).unsqueeze(0).to(device)
        pred_idx = model(x).argmax(1).item()
        class_name = class_names[pred_idx]
        comp_id = CFG.label_map[class_name]
        results.append([img_name, comp_id])

submission = pd.DataFrame(results, columns=["Id","Predicted"])
submission.to_csv("submission95.csv", index=False)
print("🎉 submission.csv saved with", len(submission), "rows")

# Distribusi prediksi
submission["Predicted"].value_counts().plot(kind="bar", rot=0)
plt.title("Distribusi Prediksi Test")
plt.show()